In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold,cross_validate
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1. EDA and basic data cleaning

In [2]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")


In [3]:
## clean unneccasry fields - customer id
df_cleaned = df.drop(columns='customerID')

In [4]:
## now the TotalCharges is str due to some missing values so first fixing that

df_cleaned['TotalCharges'] = pd.to_numeric(df_cleaned['TotalCharges'], errors='coerce').fillna(0)

In [5]:
df_cleaned['SeniorCitizen'] = df_cleaned['SeniorCitizen'].map({1:'Yes', 0: 'No'})
df_cleaned['Churn'] = df_cleaned['Churn'].map({'Yes':1,'No': 0})

## 2. Test train split

In [6]:
x = df_cleaned.drop(columns="Churn")
y = df_cleaned['Churn']
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

## 3. Preprocessing

In [7]:
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
cat_cols = x.select_dtypes(include=['object', 'str']).columns

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore",drop="if_binary"),cat_cols),
        ("num",StandardScaler(),num_cols)
    ]
)

In [9]:
pipline = Pipeline(
    steps=[
        ("preprocessing",preprocessor),
        ("model",LogisticRegression(max_iter=1000))
    ]
)

## 4. Cross-validated evaluation — pre-fit expectations

### Why am I running CV when I already have numbers from notebook 02?

Notebook 02 gave me **one** F1, **one** ROC-AUC, **one** Accuracy — from a single 80/20 split. That tells me how the model did on *that* split. It does NOT tell me whether those numbers were typical or just lucky.

CV runs the same model 5 times on 5 different (train, test) splits of the same data. So I get 5 numbers per metric, and from those I can compute:
- **Mean** = where the model usually lands
- **Std** = how much the score moves from split to split

Std is what answers "is my single-split number trustworthy?" If std is small, yes. If std is big, my single-split was a coin toss.

### What I expect

- CV **mean** should be close to the single-split numbers from `02`. If it isn't, something is wrong (different preprocessing, leak, etc.).
- CV **std** should be small — the dataset has 7043 rows, so each fold trains on ~5600 and tests on ~1400. That's plenty for a linear model to be stable.
- I added `shuffle=True, random_state=42` to `StratifiedKFold` so the folds aren't biased by whatever order the CSV was saved in.

### What "high std" would mean (didn't happen, but worth knowing)

- F1 std ≥ 0.05 → model is very sensitive to which rows are in train. Baseline is shaky.
- ROC-AUC std ≥ 0.03 → ranking ability moves around a lot. Could mean the positive class is unevenly spread across folds.

### What I'm NOT doing here

I'm not trying to *improve* the model. Same model, same preprocessing as `02`. This is purely a stability check on numbers.

In [10]:
scoring_metrics = ['f1', 'roc_auc','accuracy']
results = cross_validate(pipline, x, y, cv=skf, scoring=scoring_metrics)

In [11]:
print(f"metrics  | Mean      | Standard Deviation")
print(f"---------|-----------|-------------------")
print(f"Accuracy | {results['test_accuracy'].mean():.4f}    | {results['test_accuracy'].std():.4f}")  
print(f"F1       | {results['test_f1'].mean():.4f}    | {results['test_f1'].std():.4f}")                                                    
print(f"ROC-AUC  | {results['test_roc_auc'].mean():.4f}    | {results['test_roc_auc'].std():.4f}") 

metrics  | Mean      | Standard Deviation
---------|-----------|-------------------
Accuracy | 0.8058    | 0.0115
F1       | 0.6018    | 0.0282
ROC-AUC  | 0.8451    | 0.0134


## 5. CV vs single-split comparison

### The numbers

Single-split is from `02-baseline-logistic-regression.ipynb` (one 80/20 stratified split, `random_state=42`). CV uses 5-fold stratified, shuffled.

| Metric    | Single-split (`02`) | CV mean | CV std | Difference (single − CV mean) | Within 1 std? |
|-----------|---------------------|---------|--------|-------------------------------|---------------|
| Accuracy  | 0.8055              | 0.8058  | 0.0115 | −0.0003                       | ✅ yes        |
| F1        | 0.6040              | 0.6018  | 0.0282 | +0.0022                       | ✅ yes        |
| ROC-AUC   | 0.8420              | 0.8451  | 0.0134 | −0.0031                       | ✅ yes        |

### What this tells me — in plain words

**1. `02`'s single-split numbers were typical, not lucky.**
For all three metrics, the difference between the single-split number and the CV mean is way smaller than the std. So if I had picked a different random seed for the 80/20 split, I'd most likely have gotten a similar number.

**2. The model is stable across folds.**
Std on Accuracy is 0.0115 — that's roughly ±1 percentage point swing across folds. F1 swings a bit more (±2.8 points), which makes sense — F1 depends on both precision and recall, so it moves more than a simple accuracy count.

**3. F1 is the noisiest metric. ROC-AUC is the calmest.**
This is expected. F1 is sensitive to where the 0.5 threshold falls; ROC-AUC averages over all thresholds, so it's smoother.


### What I take into Next Notebooks

- The single-split number from `02` is a **trustworthy baseline** to compare future models against (LR-from-scratch on Saturday, DT/RF on Sunday).
- For comparing two models, if their score difference is smaller than ~2× the F1 std (≈0.056), I should run CV on both — otherwise the difference might just be noise.
- No leakage smell. F1=0.60 and Accuracy=0.81 are honest baseline numbers for an imbalanced telecom churn dataset (~26% positive class).